In [1]:
import numpy as np
import pandas as pd
import joblib
import time

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score
)

In [2]:
X_train = joblib.load("../data/X_train_zero_day.pkl")
y_train = joblib.load("../data/y_train_zero_day.pkl")

X_test_seen = joblib.load("../data/X_test_seen.pkl")
y_test_seen = joblib.load("../data/y_test_seen.pkl")

X_test_zero_day = joblib.load("../data/X_test_zero_day.pkl")
y_test_zero_day = joblib.load("../data/y_test_zero_day.pkl")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test_seen:", X_test_seen.shape)
print("y_test_seen:", y_test_seen.shape)

print("X_test_zero_day:", X_test_zero_day.shape)
print("y_test_zero_day:", y_test_zero_day.shape)

X_train: (125973, 122)
y_train: (125973,)
X_test_seen: (18794, 122)
y_test_seen: (18794,)
X_test_zero_day: (3750, 122)
y_test_zero_day: (3750,)


In [3]:
# ============================================
# Keep ONLY normal traffic for unsupervised training
# ============================================

normal_mask = (y_train == 0)

X_train_normal = X_train[normal_mask]

print("Total training samples:", len(X_train))
print("Normal training samples:", len(X_train_normal))
print("Attack training samples:", np.sum(y_train == 1))

Total training samples: 125973
Normal training samples: 67343
Attack training samples: 58630


In [4]:
# ============================================
# Validation split from NORMAL training data
# ============================================

X_normal_train, X_normal_val = train_test_split(
    X_train_normal,
    test_size=0.20,
    random_state=42
)

print("Normal training:", X_normal_train.shape)
print("Normal validation:", X_normal_val.shape)

Normal training: (53874, 122)
Normal validation: (13469, 122)


In [5]:
# ============================================
# Train Isolation Forest
# ============================================

iso_forest = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

iso_forest.fit(X_normal_train)

train_time = time.time() - start_time

print(f"Training completed in {train_time:.2f} seconds")

Training completed in 2.14 seconds


In [6]:
# ============================================
# Calculate anomaly scores
# Higher score = more anomalous
# ============================================

normal_val_scores = -iso_forest.decision_function(X_normal_val)

print("Validation score statistics:")
print("Min :", normal_val_scores.min())
print("Max :", normal_val_scores.max())
print("Mean:", normal_val_scores.mean())
print("Std :", normal_val_scores.std())

Validation score statistics:
Min : -0.1815659435961552
Max : 0.15081871283067994
Mean: -0.1262245895096455
Std : 0.057163863264451865


In [7]:
# ============================================
# Thresholds based on allowed FPR
# ============================================

fpr_targets = [0.01, 0.03, 0.05, 0.10]

thresholds = {}

for fpr in fpr_targets:
    threshold = np.quantile(
        normal_val_scores,
        1 - fpr
    )

    thresholds[fpr] = threshold

    print(
        f"Target FPR <= {fpr*100:.0f}% "
        f"-> Threshold = {threshold:.6f}"
    )

Target FPR <= 1% -> Threshold = 0.032719
Target FPR <= 3% -> Threshold = 0.006295
Target FPR <= 5% -> Threshold = -0.007979
Target FPR <= 10% -> Threshold = -0.033525


In [9]:
# ============================================
# Evaluation function
# ============================================

def evaluate_isolation_forest(
    model,
    X_seen,
    y_seen,
    X_zero_day,
    threshold
):
    
    # Scores
    seen_scores = -model.decision_function(X_seen)
    zero_day_scores = -model.decision_function(X_zero_day)

    # Predictions
    seen_pred = (seen_scores >= threshold).astype(int)
    zero_day_pred = (zero_day_scores >= threshold).astype(int)

    # ----------------------------------------
    # Seen data
    # ----------------------------------------

    cm = confusion_matrix(
        y_seen,
        seen_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) * 100

    seen_f1 = f1_score(
        y_seen,
        seen_pred,
        zero_division=0
    ) * 100

    # ----------------------------------------
    # Zero-Day detection
    # ----------------------------------------

    zero_day_detection = (
        np.sum(zero_day_pred == 1)
        / len(zero_day_pred)
        * 100
    )

    return {
        "Threshold": threshold,
        "Seen F1 (%)": seen_f1,
        "Seen FPR (%)": fpr,
        "Zero-Day Detection (%)": zero_day_detection,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn
    }

In [10]:
# ============================================
# Evaluate different FPR constraints
# ============================================

results = []

for target_fpr, threshold in thresholds.items():

    result = evaluate_isolation_forest(
        iso_forest,
        X_test_seen,
        y_test_seen,
        X_test_zero_day,
        threshold
    )

    result["Target FPR (%)"] = target_fpr * 100

    results.append(result)


iso_results = pd.DataFrame(results)

iso_results = iso_results[
    [
        "Target FPR (%)",
        "Threshold",
        "Seen F1 (%)",
        "Seen FPR (%)",
        "Zero-Day Detection (%)",
        "TP",
        "FP",
        "TN",
        "FN"
    ]
]

iso_results

,Target FPR (%),Threshold,Seen F1 (%),Seen FPR (%),Zero-Day Detection (%),TP,FP,TN,FN
0,1.0,0.032719,74.736053,1.153331,44.720000,5486,112,9599,3597
1,3.0,0.006295,77.005631,2.059520,57.973333,5812,200,9511,3271
2,5.0,-0.007979,78.223383,2.461127,65.280000,5988,239,9472,3095
3,10.0,-0.033525,79.815951,7.331892,72.400000,6505,712,8999,2578
